### **General Protocol for Automated Sequential Batch-System Molecular Docking Using AutoDock-GPU**

### **Introduction and Workflow Overview**
---
This notebook presents a **General Protocol for Automated Sequential Batch-System Molecular Docking Using AutoDock-GPU**, providing a standardized, high-throughput, and high-precision computational workflow designed to evaluate multi-ligand macromolecule interactions, binding poses, and thermodynamic affinities using **OpenCL/CUDA** GPU acceleration. Molecular docking is a fundamental *in silico* strategy in structure-based drug design, used to predict optimal binding orientations, non-covalent interaction profiles, and binding energies of small-molecule candidate libraries within a macromolecular target active site. AutoDock-GPU significantly accelerates this process by leveraging parallel GPU computing architectures, enabling extensive conformational sampling through the **Lamarckian Genetic Algorithm (LGA)**.

This notebook streamlines the entire sequential batch docking protocol, from binary environment initialization and automated AutoGrid4 spatial map calculation to GPU-accelerated sequential batch molecular docking execution, thermodynamic parsing, and publication-grade structural topology restoration, adhering strictly to computational structural biology standards. The underlying workflow executes a rigorous series of processing steps across thirteen structured procedures:

1. **Binary Environment Setup and Verification Process (Procedures 1–4):** Downloads, simplifies, grants execution permissions, and verifies the official pre-compiled **AutoDock-GPU v1.6** binary optimized for high-performance GPU execution.

2. **Workspace and Directory Initialization (Procedure 5):** Establishes dedicated input directory (**`MolecularDockingInput`**), working directory (**`Workstation`**), and output directory (**`MolecularDockingOutput`**) to isolate calculation workflows and output files.

3. **Ligand Analysis Folder Preparation and File Relocation (Procedure 6):** Dynamically initializes dedicated ligand analysis folder (**`LigandName_Analysis`**) for each prepared ligand, relocates ligand structures (**.pdbqt**) and grid parameter files (**.gpf**), and replicates the macromolecule target (**.pdbqt**) across all ligand analysis folders before removing it from the input directory.

4. **AutoGrid4 Program Installation and Batch Grid Map Calculation (Procedure 7):** Installs **AutoGrid4** into the system and executes automated grid map files generation (**.fld** and **.map**) for each ligand grid space within its respective ligand analysis folders.

5. **Macromolecule File Cleanup and Directory Relocation (Procedure 8):** Deletes duplicate macromolecule structure from all ligand analysis folders to prevent molecular docking interference and then relocates all prepared ligand analysis folder into **`Workstation`** directory for simulation readiness.

6. **Execute Sequential Production Molecular Docking Simulation and Working Directory Purging Process (Procedure 9):** Sets up **Lamarckian Genetic Algorithm** (**LGA**) search parameters using an interactive **ipywidgets** interface (**Cell 1**) and executes GPU-accelerated docking sequentially per ligand (**Cell 2**), saving simulation log files (**.dlg** and **.xml**) and top-ranked poses (**LigandName-best.pdbqt**) to **`MolecularDockingOutput`** directory while automatically purging processed ligand analysis folder to optimize runtime storage.

7. **Post-Docking Thermodynamic Analysis and Metrics Extraction (Procedure 10):** Parses simulation log file (**.dlg**) across the output directory, extracts top-ranked centroid poses (**Rank 1, Sub-Rank 1**), binding free energy ($\Delta G$), Reference RMSD ($\text{Å}$), and calculates estimated inhibition constants ($K_i$) with dynamic scientific unit scaling (**femtomolar** (**fM**) to **molar** (**M**)).

8. **Reconstruct Covalent Topology and Convert Ligand PDBQT File to PDB File Format (Procedure 11)**: Using **Open Babel** (**obabel**) to reconstruct explicit covalent bond topology (**CONECT records**) from raw lowest-energy poses (**LigandName-best.pdbqt**), yielding publication-grade ligand PDB files(**LigandName_BestPose.pdb**), while automatically applying a standardized **"_Result"** suffix to raw **LigandName.dlg** files for structured archiving.

9. **Archiving and Exporting Batch Molecular Docking Results (Procedure 12):** Compresses primary publication-ready output files (**.pdb** and **.dlg**) into a lightweight archive (**MolecularDockingOutput.zip**) for local downloading.

10. **Output Directory and ZIP Archive Purging Process (Procedure 13):** Performs optional environment maintenance by purging temporary output files and ZIP archives from the virtual machine runtime after local download.

To run this workflow, you need to provide a prepared **receptor/macromolecule structure** (**.pdbqt**), prepared **ligand structures** (**.pdbqt**), and corresponding **grid parameter files** (**.gpf**) generated from **AutoDockTools** or equivalent preparation software. Follow the sequential procedures below to execute the docking workflow.

### **Files Preparation**
---
Before initiating the sequential batch computational docking workflow, ensure you have prepared all required input files on your local workstation.

**Required Input Files:**
1. **Prepared Macromolecule File (.pdbqt):** The prepared 3D coordinate structure of the target macromolecule containing partial atomic charges and atom types (e.g., **macromolecule.pdbqt** or **receptor.pdbqt**). The filename **must contain** the keyword **"macromolecule"** or **"receptor"** (case-insensitive) to allow the automated script to distinguish it from ligand files.

2. **Prepared Ligand Files (.pdbqt):** The prepared 3D coordinate structures of single or multiple small-molecule ligands containing partial atomic charges (e.g., **Gasteiger** charges) and flexible torsional bonds in **PDBQT** format (e.g., **ligand1.pdbqt**, **ligand2.pdbqt**, etc.).

3. **Grid Parameter Files (.gpf):** The grid parameter files (**.gpf**) generated from **AutoDockTools** corresponding to each ligand (e.g., **ligand1.gpf**, **ligand2.gpf**, etc.). Each grid parameter file defines the 3D grid box coordinates, dimensions, spacing ($\text{Å}$), and atom-type map specifications required for **AutoGrid4** calculation process.

**Upload Instructions:**

You **DO NOT** need to upload these files immediately. You will be guided to upload all prepared files (**.pdbqt** and **.gpf**) directly into the designated **`MolecularDockingInput`** directory during the **Procedure 5** after directory initialization step. Once your input files reside in **`MolecularDockingInput`** directory, proceed sequentially starting with **Procedure 6**.

### **Procedure 1: Download AutoDock-GPU Executable Binary**
---
This procedure downloads the official pre-compiled Linux x64 binary of **AutoDock-GPU** (**v1.6**, **OpenCL 128 work-items**) directly from the **Center for Computational Structural Biology (CCSB) at Scripps Research** GitHub repository into the execution environment. The objective is to retrieve the core simulation engine optimized for GPU acceleration via OpenCL/CUDA architectures, ensuring high-throughput sampling capabilities. Execute this cell once at the beginning of the session to initialize the core computational engine.

To proceed, run/execute the **cell** below.

In [ ]:
# @title **Cell 1 Procedure 1**
!echo -e "\033[0;33m[INFORMATION] Downloading and Installing AutoDock-GPU...\033[0m" && wget https://github.com/ccsb-scripps/AutoDock-GPU/releases/download/v1.6/adgpu-v1.6_linux_x64_ocl_128wi -qq && echo -e "\033[0;32m[SUCCESS] AutoDock-GPU Downloading and Installing Process Complete.\033[0m" || echo -e "\033[0;31m[ERROR] AutoDock-GPU Downloading and Installing Process Failed.\033[0m"

### **Procedure 2: Simplify Executable Binary Filename**
---
This procedure renames the downloaded long binary file (**adgpu-v1.6_linux_x64_ocl_128wi**) to a simplified executable alias (**adgpu**). The objective is to streamline command-line interaction, making downstream execution scripts cleaner, more readable, and less prone to typographical errors during manual or automated invocation.

To proceed, run/execute the **cell** below.

In [ ]:
# @title **Cell 1 Procedure 2**
!echo -e "\033[0;33m[INFORMATION] Renaming AutoDock-GPU Binary File...\033[0m" && mv adgpu-v1.6_linux_x64_ocl_128wi adgpu && echo -e "\033[0;32m[SUCCESS] AutoDock-GPU Binary File Renaming Process Complete.\033[0m" || echo -e "\033[0;31m[ERROR] Binary File Renaming Process Failed.\033[0m"

### **Procedure 3: Grant Binary Execution Permissions**
---
This procedure modifies file permissions using the **chmod** command to grant execution rights to the **adgpu** binary. In Unix-like Linux environments, newly downloaded external binaries lack execution privileges by default for security reasons. Granting read, write, and execute permissions (**755**) ensures the operating system permits the simulation engine to run as an active process without encountering **"Permission Denied"** runtime errors.

To proceed, run/execute the **cell** below.

In [ ]:
# @title **Cell 1 Procedure 3**
!chmod 755 -R /content/adgpu && echo -e "\033[0;32m[SUCCESS] AutoDock-GPU Binary Execution Permissions Granted.\033[0m" || echo -e "\033[0;31m[ERROR] AutoDock-GPU Binary Execution Permissions Denied.\033[0m"

### **Procedure 4: Verify Binary Installation and Display Usage Reference**
---
This procedure executes the **adgpu** binary without command-line arguments to confirm that the compiled program runs correctly within the current GPU runtime. The objective is to verify binary integrity, print the software version, and display the official command-line reference menu. This allows users to inspect available algorithm parameters, execution flags, and default settings prior to initiating simulation runs.

To proceed, run/execute the **cell** below.

In [ ]:
# @title **Cell 1 Procedure 4**
!./adgpu && echo -e "\033[0;32m[SUCCESS] Binary Installation Verification Completed, Usage Guide is Displayed.\033[0m" || echo -e "\033[0;31m[ERROR] Binary Installation Verification Failed, Usage Guide is Not Displayed.\033[0m"

### **Procedure 5: Workspace and Directory Initialization**
---
This procedure initializes the structured workspace environment required for the automated batch-system molecular docking workflow. Executing the cell below creates three dedicated directories to segregate input, processing, and output files:

1. **Input Directory (`MolecularDockingInput`):** Acts as the central repository for all raw input files uploaded from your local workstation, including all prepared 3D ligand structures (**.pdbqt**), grid parameter files (**.gpf**), and the target macromolecule structure (**.pdbqt**).

2. **Working Directory (`Workstation`):** Serves as the isolated execution workspace where individual ligand analysis folder (**`LigandName_Analysis`**) are established to run AutoGrid4 grid energy calculations and AutoDock-GPU docking simulations without file conflict.

3. **Output Directory (`MolecularDockingOutput`):** Consolidates all final molecular docking outputs, including simulation output logs (**.dlg**), XML files (**.xml**), and top-ranked centroid binding poses files (**LigandName-best.pdbqt**).

To proceed, run/execute the **cell** below **FIRST** to create all three directories. Once initialized, transfer all your prepared input files directly into the **`MolecularDockingInput`** directory prior to running **Procedure 6** by locate the **`MolecularDockingInput`** directory, click the three vertical dots icon, select **"Upload"** option, and then select all the required files as previously specified.

In [ ]:
# @title **Cell 1 Procedure 5**
!mkdir -p /content/MolecularDockingInput && echo -e "\033[0;32m[SUCCESS] Input Directory (MolecularDockingInput) Initialization Complete.\033[0m" || echo -e "\033[0;31m[ERROR] Input Directory (MolecularDockingInput) Initialization Failed.\033[0m" && mkdir -p /content/Workstation && echo -e "\033[0;32m[SUCCESS] Working Directory (Workstation) Initialization Complete.\033[0m" || echo -e "\033[0;31m[ERROR] Working Directory (Workstation) Initialization Failed.\033[0m" && mkdir -p /content/MolecularDockingOutput && echo -e "\033[0;32m[SUCCESS] Output Directory (MolecularDockingOutput) Initialization Complete.\033[0m" || echo -e "\033[0;31m[ERROR] Output Directory (MolecularDockingOutput) Initialization Failed.\033[0m"

### **Procedure 6: Ligand Analysis Folder Preparation and File Relocation**
---
This procedure organizes the uploaded input files inside **`MolecularDockingInput`** directory into isolated subdirectories to enable parallel, batch-system processing. Executing the three code cells below performs the following sequential preparation steps:

1. **Analysis Folder Creation (Cell 1):** Scans **`MolecularDockingInput`** for small-molecule ligand files (**.pdbqt**) and automatically generates a dedicated working ligand analysis folder (**`LigandName_Analysis`**) for each identified ligand.

2. **Ligand and Grid Parameter File Relocation (Cell 2):** Moves each ligand structure (**.pdbqt**) and its matching grid parameter file (**.gpf**) into its respective **`LigandName_Analysis`** folder.

3. **Macromolecule Distribution (Cell 3):** Identifies the target macromolecule structure (**.pdbqt**), replicates it across all **`LigandName_Analysis`** folders to ensure each workspace is fully self-contained for grid generation, and cleans up the original file from the input directory.

To proceed, run/execute the **three cells below sequentially** to complete the input file preparation before proceeding to **AutoGrid4** calculation process.

In [ ]:
# @title **Cell 1 Procedure 6**
import glob
import os

input_dir = "/content/MolecularDockingInput"

if not os.path.exists(input_dir):
  print(f"\033[0;31m[ERROR] Input Directory Not Found.\033[0m")
else:
  all_pdbqt = glob.glob(os.path.join(input_dir, "*.pdbqt"))

  ligand_files = [
      f
      for f in all_pdbqt
      if "macromolecule" not in os.path.basename(f).lower()
      and "receptor" not in os.path.basename(f).lower()
  ]

  if not ligand_files:
    print(
        "\033[0;31m[ERROR] PDBQT Ligand File Not Found in Input Directory.\033[0m"
    )
  else:
    created_count = 0
    print("\033[0;33m[INFORMATION] Preparing Ligand Analysis Folder in the Input Directory...\033[0m\n")

    for lfile in sorted(ligand_files):
      ligand_name = os.path.splitext(os.path.basename(lfile))[0]
      folder_name = f"{ligand_name}_Analysis"
      folder_path = os.path.join(input_dir, folder_name)

      os.makedirs(folder_path, exist_ok=True)
      print(f"\033[0;32m[SUCCESS] Folder Created: {folder_name}\033[0m")
      created_count += 1

    print("-" * 80)
    print(
        f"\033[0;32m[SUCCESS] Total {created_count} Ligand Analysis Folder Preparation Complete.\033[0m"
    )

In [ ]:
# @title **Cell 2 Procedure 6**
import glob
import os
import shutil

input_dir = "/content/MolecularDockingInput"

if not os.path.exists(input_dir):
  print(f"\033[0;31m[ERROR] Input Directory Not Found.\033[0m")
else:
  all_pdbqt = glob.glob(os.path.join(input_dir, "*.pdbqt"))
  ligand_files = [
      f
      for f in all_pdbqt
      if "macromolecule" not in os.path.basename(f).lower()
      and "receptor" not in os.path.basename(f).lower()
  ]

  if not ligand_files:
    print(
        "\033[0;31m[ERROR] Ligand PDBQT File Not Found in the Input Directory.\033[0m"
    )
  else:
    moved_count = 0
    print(
        "\033[0;33m[INFORMATION] Relocating Each Ligand PDBQT File and Grid Parameter File (GPF) to its Respective Folder...\033[0m\n"
    )

    for lfile in sorted(ligand_files):
      filename = os.path.basename(lfile)
      ligand_name = os.path.splitext(filename)[0]
      target_folder = os.path.join(input_dir, f"{ligand_name}_Analysis")

      if os.path.exists(target_folder):
        shutil.move(lfile, os.path.join(target_folder, filename))

        gpf_file = os.path.join(input_dir, f"{ligand_name}.gpf")
        if os.path.exists(gpf_file):
          shutil.move(
              gpf_file, os.path.join(target_folder, f"{ligand_name}.gpf")
          )
          print(
              f"\033[0;32m[SUCCESS] {ligand_name} PDBQT File and Grid Parameter File (GPF) Has Been Relocated to Ligand Analysis Folder ({ligand_name}_Analysis).\033[0m"
          )
        else:
          print(
              f"\033[0;32m[SUCCESS] {ligand_name} PDBQT File Has Been Relocated to Ligand Analysis Folder ({ligand_name}_Analysis), Grid Parameter File (GPF) Not Found.\033[0m"
          )

        moved_count += 1
      else:
        print(
            f"\033[0;31m[WARNING] Ligand Analysis Folder ({ligand_name}_Analysis) Not Found.\033[0m"
        )

    print("-" * 80)
    print(
        f"\033[0;32m[SUCCESS] Relocation Process for {moved_count} Ligand Complete.\033[0m"
    )

In [ ]:
# @title **Cell 3 Procedure 6**
import glob
import os
import shutil

input_dir = "/content/MolecularDockingInput"

if not os.path.exists(input_dir):
  print(f"\033[0;31m[ERROR] Input Directory Not Found.\033[0m")
else:
  all_pdbqt = glob.glob(os.path.join(input_dir, "*.pdbqt"))
  macro_files = [
      f
      for f in all_pdbqt
      if "macromolecule" in os.path.basename(f).lower()
      or "receptor" in os.path.basename(f).lower()
  ]

  if not macro_files and all_pdbqt:
    macro_files = all_pdbqt

  if not macro_files:
    print(
        "\033[0;31m[ERROR] Macromolecule PDBQT File Not Found in the Input Directory.\033[0m"
    )
  else:
    macro_path = macro_files[0]
    macro_name = os.path.basename(macro_path)

    analysis_folders = [
        os.path.join(input_dir, d)
        for d in os.listdir(input_dir)
        if os.path.isdir(os.path.join(input_dir, d)) and d.endswith("_Analysis")
    ]

    if not analysis_folders:
      print(
          "\033[0;31m[ERROR] Ligand Analysis Folder Not Found in the Input Directory.\033[0m"
      )
    else:
      copied_count = 0
      print(f"\033[0;33m[INFORMATION] Copying Macromolecule PDBQT File ({macro_name}) to All Ligand Analysis Folder...\033[0m\n")

      for folder in sorted(analysis_folders):
        target_path = os.path.join(folder, macro_name)
        shutil.copy2(macro_path, target_path)
        folder_name = os.path.basename(folder)
        print(
            f"\033[0;32m[SUCCESS] Macromolecule PDBQT File ({macro_name}) Has Been Copied to Ligand Analysis Folder ({folder_name}).\033[0m"
        )
        copied_count += 1

      os.remove(macro_path)

      print("-" * 80)
      print(f"\033[0;32m[SUCCESS] Macromolecule PDBQT File Has Been Copied to {copied_count} Ligand Analysis Folder, Original Macromolecule PDBQT File ({macro_name}) in the Input Directory Has Been Deleted.\033[0m")

### **Procedure 7: AutoGrid4 Program Installation and Batch Grid Map Calculation**
---
This procedure handles the system installation of **AutoGrid4** program and automates the spatial grid map calculations for all ligand analysis datasets created in **Procedure 6**. Executing the three code cells below completes the following sequential tasks:

1. **AutoGrid4 Program Installation (Cell 1):** Downloads and installs the **AutoGrid4** suite into the Linux runtime environment using system package management.

2. **Program Installation Verification (Cell 2):** Verifies proper installation by displaying the binary version information.

3. **Automated Batch Grid Generation (Cell 3):** Iterates through each **`LigandName_Analysis`** folder inside **`MolecularDockingInput`** directory, locates the respective grid parameter file (**.gpf**) and target macromolecule structure (**.pdbqt**), then executes **AutoGrid4** program to compute **atomic affinity grid maps** (**.map**), **desolvation and electrostatic maps** (**.d.map** and **.e.map**), and the **grid map descriptor** (**.fld**).

To proceed, run/execute the **three cells below sequentially** to complete the grid map calculations before proceeding to **Procedure 8**.

In [ ]:
# @title **Cell 1 Procedure 7**
!echo -e "\033[0;33m[INFORMATION] Installing AutoGrid4 Program into the System...\033[0m" && \apt-get update -qq > /dev/null 2>&1 && \apt-get install -y autogrid > /dev/null 2>&1 && \ln -sf /usr/bin/autogrid4/usr/local/bin/autogrid4 && \echo -e "\033[0;32m[SUCCESS] AutoGrid4 Program Installation Complete.\033[0m" || \echo -e "\033[0;31m[ERROR] AutoGrid4 Program Installation Failed.\033[0m"

In [ ]:
# @title **Cell 2 Procedure 7**
!autogrid4 --version && echo -e "\033[0;32m[SUCCESS] AutoGrid4 Program Installation Verification Complete, Information is Displayed.\033[0m" || echo -e "\033[0;31m[ERROR] AutoGrid4 Installation Verification Failed, Information is not Displayed.\033[0m"

In [ ]:
# @title **Cell 3 Procedure 7**
import glob
import os
import subprocess

input_dir = "/content/MolecularDockingInput"
analysis_folders = sorted(glob.glob(os.path.join(input_dir, "*_Analysis")))

if not analysis_folders:
  print(
      "\033[0;31m[ERROR] Ligand Analysis Folder Not Found in the Input Directory.\033[0m"
  )
else:
  print(
      "\033[0;33m[INFORMATION] AutoGrid4 Grid Map Calculation in Process...\033[0m\n"
  )
  success_count = 0

  for folder in analysis_folders:
    folder_name = os.path.basename(folder)
    gpf_files = glob.glob(os.path.join(folder, "*.gpf"))

    if not gpf_files:
      print(
          f"\033[0;31m[ERROR] Grid Parameter File (GPF) Not Found in the {folder_name} Folder.\033[0m"
      )
      continue

    gpf_file = os.path.basename(gpf_files[0])
    glg_file = f"{os.path.splitext(gpf_file)[0]}.glg"

    cmd = f'autogrid4 -p "{gpf_file}" -l "{glg_file}"'
    res = subprocess.run(
        cmd, shell=True, cwd=folder, capture_output=True, text=True
    )

    if res.returncode == 0:
      print(
          f"\033[0;32m[SUCCESS] AutoGrid4 Grid Map Calculation Complete for Ligand Analysis Folder ({folder_name}).\033[0m"
      )
      success_count += 1
    else:
      print(f"\033[0;31m[ERROR] AutoGrid4 Grid Map Calculation Failed for Ligand Analysis Folder ({folder_name}).\033[0m")

  print("-" * 80)
  if success_count > 0:
    print(
        f"\033[0;32m[SUCCESS] AutoGrid4 Grid Map Calculation Complete for {success_count} Grid Parameter File (GPF) in the {success_count} Ligand Analysis Folder.\033[0m"
    )
  else:
    print(
        "\033[0;31m[ERROR] No Grid Parameter File (GPF) Found or AutoGrid4 Grid Map Calculation Failed.\033[0m"
    )

### **Procedure 8: Macromolecule File Cleanup and Directory Relocation**
---
This procedure cleans up redundant temporary files and transfers all prepared ligand workspaces into the central execution directory. Executing the two code cells below performs the following sequential tasks:

1. **Macromolecule File Cleanup (Cell 1):** Deletes macromolecule PDBQT file from each **`LigandName_Analysis`** folder, as grid calculations are complete and these macromolecule PDBQT file are no longer required for AutoDock-GPU.

2. **Workstation Relocation (Cell 2):** Relocates all fully prepared **`LigandName_Analysis`** folders from **`MolecularDockingInput`** directory into the central **`Workstation`** directory to prepare for sequential batch molecular docking executions.

To proceed, run/execute the **two cells below sequentially** to complete the workspace preparation before proceeding to **Procedure 9**.

In [ ]:
# @title **Cell 1 Procedure 8**
import glob
import os

input_dir = "/content/MolecularDockingInput"
analysis_folders = sorted(glob.glob(os.path.join(input_dir, "*_Analysis")))

if not analysis_folders:
  print(
      f"\033[0;31m[ERROR] Ligand Analysis Folder Not Found in the Input Directory.\033[0m"
  )
else:
  deleted_count = 0
  print(
      "\033[0;33m[INFORMATION] Deleting Macromolecule PDBQT File From All Ligand Analysis Folder...\033[0m\n"
  )

  for folder in analysis_folders:
    folder_name = os.path.basename(folder)

    pdbqt_files = glob.glob(os.path.join(folder, "*.pdbqt"))
    macro_files = [
        f
        for f in pdbqt_files
        if "macromolecule" in os.path.basename(f).lower()
        or "receptor" in os.path.basename(f).lower()
    ]

    for macro_file in macro_files:
      macro_name = os.path.basename(macro_file)
      os.remove(macro_file)
      print(
          f"\033[0;32m[SUCCESS] Macromolecule PDBQT File ({macro_name}) Has Been Deleted From Ligand Analysis Folder ({folder_name}).\033[0m"
      )
      deleted_count += 1

  print("-" * 80)
  print(
      f"\033[0;32m[SUCCESS] Deleted {deleted_count} Macromolecule PDBQT File From All Ligand Analysis Folder.\033[0m"
  )

In [ ]:
# @title **Cell 2 Procedure 8**
import glob
import os
import shutil

input_dir = "/content/MolecularDockingInput"
workstation_dir = "/content/Workstation"

os.makedirs(workstation_dir, exist_ok=True)

analysis_folders = sorted(glob.glob(os.path.join(input_dir, "*_Analysis")))

if not analysis_folders:
  print(
      f"\033[0;31m[ERROR] Ligand Analysis Folder Not Found in the Input Directory.\033[0m"
  )
else:
  moved_count = 0
  print(
      "\033[0;33m[INFORMATION] Relocating All Ligand Analysis Folder to the Working Directory...\033[0m\n"
  )

  for folder in analysis_folders:
    folder_name = os.path.basename(folder)
    target_path = os.path.join(workstation_dir, folder_name)

    if os.path.exists(target_path):
      shutil.rmtree(target_path)

    shutil.move(folder, target_path)
    print(
        f"\033[0;32m[SUCCESS] Ligand Analysis Folder ({folder_name}) Has Been Relocated to the Working Directory.\033[0m"
    )
    moved_count += 1

  print("-" * 80)
  print(
      f"\033[0;32m[SUCCESS] Relocated {moved_count} Ligand Analysis Folder to the Working Directory.\033[0m"
  )

### **Procedure 9: Execute Sequential Production Molecular Docking Simulation and Working Directory Purging Process**

This procedure executes GPU-accelerated molecular docking simulations using **AutoDock-GPU** through a modular two-cell workflow designed for execution control and workspace storage optimization:

1. **Lamarckian Genetic Algorithm Parameters Setup (Cell 1):** Provides an interactive widget interface to configure **Lamarckian Genetic Algorithm** **(LGA)** parameters, including number of lamarckian genetic algorithm runs (**nrun**), maximum number of evaluations (**nev**), population size (**psize**), and maximum number of generations (**ngen**), and then saves them directly into **Jupyter kernel memory**.

2. **Sequential Single-Ligand Docking and Working Directory Purging (Cell 2):** Processes one ligand analysis folder (**`LigandName_Analysis`**) in one cell run/execution, runs the AutoDock-GPU engine (**`adgpu`**), automatically transfers result files (**.dlg**, **.xml**, and **LigandName-best.pdbqt**) to the **`MolecularDockingOutput`** directory, and purges the processed ligand analysis folder upon completion to maintain a clean workspace.

To proceed, run/execute **Cell 1** once to configure and save your search parameters into Jupyter kernel memory, then run/execute **Cell 2** repeatedly, once for each prepared ligand. For instance, if you have 10 ligand analysis folders in the workstation, you will need to run/execute **Cell 2** 10 times to process each docking simulation until all ligand analysis folders are completed and cleared from working directory (**`Workstation`**).

In [ ]:
# @title **Cell 1 Procedure 9**
from IPython.display import clear_output, display
import ipywidgets as widgets

css_style = widgets.HTML(
    value="""
<style>
    .widget-label {
        min-width: 280px !important;
        max-width: 280px !important;
        text-align: left !important;
    }
    .widget-text input, .widget-inttext input {
        background-color: #383838 !important;
        color: #ffffff !important;
        border: 1px solid #5a5a5a !important;
        border-radius: 2px !important;
    }
</style>
"""
)

title_w = widgets.HTML(
    value=(
        "<h3 style='margin-top:0px; margin-bottom:15px;"
        " font-weight:bold;'>Lamarckian Genetic Algorithm (LGA) Parameters</h3>"
    )
)

label_style = {"description_width": "280px"}
widget_layout = widgets.Layout(width="520px")

nrun_w = widgets.IntText(value=100, description="Number of Lamarckian Genetic Algorithm Runs:", style=label_style, layout=widget_layout)
nev_w = widgets.IntText(value=25000000, description="Maximum Number of Evaluations:", style=label_style, layout=widget_layout)
psize_w = widgets.IntText(value=300, description="Population Size:", style=label_style, layout=widget_layout)
ngen_w = widgets.IntText(value=50000, description="Maximum Number of Generations:", style=label_style, layout=widget_layout)

save_button = widgets.Button(
    description="Save Lamarckian Genetic Algorithm Parameters",
    layout=widgets.Layout(width="520px", height="40px", border="1px solid #5a5a5a"),
)
save_button.style.button_color = "#383838"
save_button.style.text_color = "#ffffff"

output_area = widgets.Output()

def save_parameters(b):
    with output_area:
        clear_output()
        print("\033[0;32m[SUCCESS] Lamarckian Genetic Algorithm Parameters Saved.\033[0m")

save_button.on_click(save_parameters)

display(
    widgets.VBox([
        css_style, title_w, nrun_w, nev_w, psize_w, ngen_w,
        widgets.HTML(value="<br>"), save_button,
        widgets.HTML(value="<br>"), output_area
    ])
)

In [ ]:
# @title **Cell 2 Procedure 9**
import glob
import os
import pty
import shutil
import subprocess
from IPython.display import clear_output

work_dir = "/content/Workstation"
output_dir = "/content/MolecularDockingOutput"
os.makedirs(output_dir, exist_ok=True)

if 'nrun_w' not in globals():
    print("\033[0;31m[ERROR] Lamarckian Genetic Algorithm Parameters Not Found. Please Run Cell 1 Procedure 9 First.\033[0m")
else:
    analysis_folders = sorted(glob.glob(os.path.join(work_dir, "*_Analysis")))

    if not analysis_folders:
        print("\033[0;31m[ERROR] No Ligand Analysis Folder Found in the Workstation Directory, Molecular Docking Simulation Cannot be Executed.\033[0m")
    else:
        folder_path = analysis_folders[0]
        folder_name = os.path.basename(folder_path)
        ligand_name = folder_name.replace("_Analysis", "")

        adgpu_bin = None
        possible_bins = ["/content/adgpu", "/content/Workstation/adgpu", shutil.which("autodock_gpu"), shutil.which("adgpu"), "/usr/local/bin/autodock_gpu", "/usr/local/bin/adgpu"]
        for b_path in possible_bins:
            if b_path and os.path.exists(b_path):
                adgpu_bin = b_path
                break
        if not adgpu_bin:
            adgpu_bin = "autodock_gpu"

        fld_files = glob.glob(os.path.join(folder_path, "*.fld"))
        pdbqt_files = glob.glob(os.path.join(folder_path, "*.pdbqt"))
        ligand_files = [f for f in pdbqt_files if not f.endswith("-best.pdbqt") and not f.endswith("_best.pdbqt")]

        if not fld_files or not ligand_files:
            print(f"\033[0;31m[ERROR] Missing Grid Map Descriptor File (FLD) or Ligand PDBQT File in Ligand Analysis Folder ({folder_name}).\033[0m")
        else:
            fld_basename = os.path.basename(fld_files[0])
            lfile_basename = os.path.basename(ligand_files[0])

            cmd = (
                f'"{adgpu_bin}" --ffile "{fld_basename}" --lfile "{lfile_basename}" '
                f'--nrun {nrun_w.value} --nev {nev_w.value} --psize {psize_w.value} --ngen {ngen_w.value} '
                '--autostop 0 --heuristics 0 --gbest 1 '
                f'--resnam "{ligand_name}"'
            )

            master, slave = pty.openpty()
            proc = subprocess.Popen(cmd, shell=True, cwd=folder_path, stdout=slave, stderr=slave, stdin=subprocess.DEVNULL, close_fds=True)
            os.close(slave)

            full_output = ""
            while True:
                try:
                    data = os.read(master, 1024)
                    if not data: break
                    full_output += data.decode("utf-8", errors="ignore")
                    clear_output(wait=True)
                    print(f"\033[0;33m[INFORMATION] Processing Molecular Docking Simulation for {ligand_name}...\033[0m\n")
                    print(full_output, end="", flush=True)
                except OSError:
                    break

            os.close(master)
            proc.wait()

            expected_dlg = os.path.join(folder_path, f"{ligand_name}.dlg")

            if proc.returncode == 0 and os.path.exists(expected_dlg):
                for ext in [".dlg", ".xml"]:
                    src = os.path.join(folder_path, f"{ligand_name}{ext}")
                    if os.path.exists(src): shutil.copy2(src, os.path.join(output_dir, f"{ligand_name}{ext}"))

                best_pdbqt_src = None
                for b_name in [f"{ligand_name}-best.pdbqt", f"{ligand_name}_best.pdbqt", f"{ligand_name}.pdbqt"]:
                    candidate = os.path.join(folder_path, b_name)
                    if os.path.exists(candidate) and candidate != os.path.join(folder_path, lfile_basename):
                        best_pdbqt_src = candidate
                        break

                if not best_pdbqt_src:
                    for f in glob.glob(os.path.join(folder_path, "*.pdbqt")):
                        if os.path.basename(f) != lfile_basename:
                            best_pdbqt_src = f
                            break

                if best_pdbqt_src: shutil.copy2(best_pdbqt_src, os.path.join(output_dir, f"{ligand_name}-best.pdbqt"))

                shutil.rmtree(folder_path)

                clear_output(wait=True)
                print(f"\033[0;32m[SUCCESS] {ligand_name} Molecular Docking Simulation Complete, Related Ligand Analysis Folder Removed from Working Directory.\033[0m")
            else:
                clear_output(wait=True)
                print(f"\033[0;31m[ERROR] {ligand_name} Molecular Docking Simulation Failed.\033[0m")

### **Procedure 10: Post-Docking Thermodynamic Analysis and Metrics Extraction**

This procedure performs an automated, batch-wide post-docking evaluation across all simulation output logs (**.dlg**) stored in the **`MolecularDockingOutput`** directory. By systematically parsing the **CLUSTERING HISTOGRAM** table at the **RANKING** entry for every docked ligand in the batch, the Python script accurately pinpoints each compound's top-ranked centroid pose (**Rank 1, Sub-Rank 1**) along with its corresponding **Run** and **Cluster** designations, ensuring reproducible validation and high-throughput evaluation against raw simulation logs.

**Thermodynamic Metrics and Calculations:**

1. **Binding Free Energy ($\Delta G$):** Extracted directly from the top-ranked energy pose for each ligand in $\text{kcal/mol}$.

2. **Inhibition Constant ($K_i$):** Calculated using the standard thermodynamic relation $K_i = e^{\frac{\Delta G}{RT}}$, where $R$ is the universal gas constant ($0.00198719\text{ kcal}\cdot\text{mol}^{-1}\cdot\text{K}^{-1}$) and $T$ is standard room temperature ($298.15\text{ K}$).

3. **Dynamic Unit Scaling:** The script automatically formats calculated $K_i$ values into standard scientific concentration units ranging from **femtomolar** (**fM**), **picomolar** (**pM**), **nanomolar** (**nM**), **micromolar** ($\mu\$**M**), **millimolar** (**mM**), to **molar** (**M**) for each ligand.

4. **Reference RMSD:** Parses the **Root-Mean-Square Deviation** relative to the reference binding conformation for immediate redocking fidelity verification across the dataset.

To proceed, run/execute the **cell** below.

In [ ]:
# @title **Cell 1 Procedure 10**
import glob
import math
import os
import re

output_dir = "/content/MolecularDockingOutput"
dlg_files = sorted(glob.glob(os.path.join(output_dir, "*.dlg")))

if not dlg_files:
  print(
      f"\033[0;31m[ERROR] DLG File Not Found in the Output Directory.\033[0m"
  )
else:
  print(
      f"\033[0;33m[INFORMATION] Processing {len(dlg_files)} DLG File for Ligand Best Pose Analysis...\033[0m\n"
  )

  for dlg_file in dlg_files:
    ligand_name = os.path.splitext(os.path.basename(dlg_file))[0]

    best_cluster = None
    best_rank = None
    best_sub_rank = None
    best_run = None
    best_dG = None
    best_rmsd = None

    try:
      with open(dlg_file, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

      in_histogram = False
      for line in lines:
        line_lower = line.lower()

        if any(
            k in line_lower
            for k in [
                "clustering histogram",
                "clustering_histogram",
                "rmsd table",
            ]
        ):
          in_histogram = True
          continue

        if in_histogram:
          if any(
              w in line_lower
              for w in [
                  "tolerance",
                  "grid",
                  "matrix",
                  "version",
                  "autodock",
                  "copyright",
                  "binding energy",
                  "sub-rank",
              ]
          ):
            continue

          raw_nums = re.findall(r"[-+]?\d+\.?\d*", line)
          nums = []
          for n in raw_nums:
            try:
              nums.append(float(n))
            except ValueError:
              pass

          if len(nums) >= 5:
            if int(nums[0]) == 1 and int(nums[1]) == 1:
              if len(nums) == 6:
                c, r, sub, run, dg, rmsd = (
                    1,
                    1,
                    1,
                    int(nums[2]),
                    nums[3],
                    nums[5],
                )
              elif len(nums) >= 7:
                c, r, sub, run, dg, rmsd = (
                    int(nums[0]),
                    int(nums[1]),
                    int(nums[2]),
                    int(nums[3]),
                    nums[4],
                    nums[6],
                )
              else:
                c, r, sub, run, dg, rmsd = (
                    1,
                    1,
                    1,
                    int(nums[2]),
                    nums[3],
                    nums[4],
                )

              if dg < 0:
                best_cluster, best_rank, best_sub_rank, best_run, best_dG, (
                    best_rmsd
                ) = (c, r, sub, run, dg, rmsd)
                break

      if best_dG is None:
        for line in lines:
          line_lower = line.lower()
          if any(
              w in line_lower
              for w in [
                  "tolerance",
                  "grid",
                  "matrix",
                  "version",
                  "autodock",
                  "copyright",
                  "time",
                  "about",
              ]
          ):
            continue

          raw_nums = re.findall(r"[-+]?\d+\.?\d*", line)
          nums = []
          for n in raw_nums:
            try:
              nums.append(float(n))
            except ValueError:
              pass

          if len(nums) >= 5:
            if int(nums[0]) == 1 and int(nums[1]) == 1:
              if len(nums) == 6:
                c, r, sub, run, dg, rmsd = (
                    1,
                    1,
                    1,
                    int(nums[2]),
                    nums[3],
                    nums[5],
                )
              elif len(nums) >= 7:
                c, r, sub, run, dg, rmsd = (
                    int(nums[0]),
                    int(nums[1]),
                    int(nums[2]),
                    int(nums[3]),
                    nums[4],
                    nums[6],
                )
              else:
                c, r, sub, run, dg, rmsd = (
                    1,
                    1,
                    1,
                    int(nums[2]),
                    nums[3],
                    nums[4],
                )

              if dg < 0:
                best_cluster, best_rank, best_sub_rank, best_run, best_dG, (
                    best_rmsd
                ) = (c, r, sub, run, dg, rmsd)
                break

      if best_dG is None:
        for line in lines:
          if (
              "Estimated Free Energy of Binding" in line
              or "Binding Energy" in line
          ):
            matches = re.findall(r"[-+]?\d+\.\d+", line)
            if matches:
              val = float(matches[0])
              if val < 0:
                best_dG = val
                best_cluster, best_rank, best_sub_rank, best_run, best_rmsd = (
                    1,
                    1,
                    1,
                    1,
                    0.0,
                )
                break

    except Exception:
      pass

    print("=" * 70)
    print(f"Ligand: {ligand_name}")

    if best_dG is not None:
      ki = math.exp(best_dG / (0.00198719 * 298.15))

      if ki < 1e-12:
        parameter, unit = ki * 1e15, "fM (femtomolar)"
      elif ki < 1e-9:
        parameter, unit = ki * 1e12, "pM (picomolar)"
      elif ki < 1e-6:
        parameter, unit = ki * 1e9, "nM (nanomolar)"
      elif ki < 1e-3:
        parameter, unit = ki * 1e6, "uM (micromolar)"
      elif ki < 1:
        parameter, unit = ki * 1e3, "mM (millimolar)"
      else:
        parameter, unit = ki, "M (molar)"

      print(f"Best Molecular Docking Pose: Cluster {best_cluster}, Rank {best_rank}, Sub-Rank {best_sub_rank}, Run {best_run}")
      print(f"Binding Energy: {best_dG:6.2f} kcal/mol")
      print(f"Estimated Inhibition Constant (Ki): {parameter:6.2f} {unit}")
      if best_rmsd is not None:
        print(f"Reference RMSD: {best_rmsd:6.2f} Å")
    else:
      print("\033[0;31m[ERROR] Data Not Found in the DLG File.\033[0m")
    print("=" * 70 + "\n")

### **Procedure 11: Reconstruct Covalent Topology and Convert Ligand PDBQT File to PDB File Format**
---
This procedure automates the batch-wide structural restoration and conversion of optimal docked poses from raw PDBQT format into standardized, publication-grade PDB files. Executing the three code cells below performs the following sequential tasks:

1. **Open Babel Tool Installation (Cell 1):** Installs the **Open Babel** (**obabel**) suite into the Linux runtime environment using system package management.

2. **Topological Reconstruction and PDB Conversion (Cell 2):** Parses all raw **LigandName-best.pdbqt** files in batch, recalculates 3D interatomic bond geometries, and reconstructs fully intact, standardized PDB files (**LigandName_BestPose.pdb**) with explicit covalent bond topology (**CONECT records**).

3. **Molecular Docking Log File Renaming (Cell 3):** Applies a standardized **"_Result"** suffix system to all raw simulation log files (**.dlg**) in the **`MolecularDockingOutput`** directory, renaming them from **LigandName.dlg** to **LigandName_Result.dlg** for structured archiving and post-docking identification.

Raw **.pdbqt** files generated directly by AutoDock-GPU frequently lack explicit **covalent bond topology** (**CONECT records**) and **atomic connectivity parameters**. Consequently, opening an unrestored **LigandName-best.pdbqt** file in molecular visualization software, such as **BIOVIA Discovery Studio Visualizer**, **PyMOL**, or **UCSF Chimera**, often results in graphical artifacts, missing covalent bonds, or fragmented atomic representations.

This automated topology reconstruction ensures all covalent bonds and atomic connectivity networks are accurately restored across the dataset, yielding refined structures optimized for downstream 2D/3D protein-ligand interaction profiling.

To proceed, run/execute the **three cells below sequentially** to complete the topological reconstruction and file renaming before proceeding to **Procedure 12**.

In [ ]:
# @title **Cell 1 Procedure 11**
!echo -e "\033[0;33m[INFORMATION] Installing Open Babel...\033[0m" && apt-get install -y openbabel -qq >/dev/null 2>&1 && echo -e "\033[0;32m[SUCCESS] Open Babel Installation Completed.\033[0m" || echo -e "\033[0;31m[ERROR] Open Babel Installation Failed.\033[0m"

In [ ]:
# @title **Cell 2 Procedure 11**
!cd /content/MolecularDockingOutput && ls *-best.pdbqt > /dev/null 2>&1 && echo -e "\033[0;33m[INFORMATION] Converting All Available Ligand PDBQT File in the Output Directory to PDB File Format...\033[0m" && count=0 && for f in *-best.pdbqt; do obabel -ipdbqt "$f" -opdb -O "${f%-best.pdbqt}_BestPose.pdb" > /dev/null 2>&1 && count=$((count+1)); done && echo -e "\033[0;32m[SUCCESS] $count Ligand PDBQT File Converted to PDB File Format.\033[0m" || echo -e "\033[0;31m[ERROR] Failed Converting Ligand PDBQT File to PDB File Format.\033[0m"

In [ ]:
# @title **Cell 3 Procedure 11**
!cd /content/MolecularDockingOutput && ls *.dlg > /dev/null 2>&1 && echo -e "\033[0;33m[INFORMATION] Renaming DLG Files to Include the Suffix System...\033[0m" && count=0 && for f in *.dlg; do if [[ "$f" != *"_Result.dlg" ]]; then mv "$f" "${f%.dlg}_Result.dlg" && count=$((count+1)); fi; done && echo -e "\033[0;32m[SUCCESS] $count DLG File Renaming Process to Include Suffix System Complete.\033[0m" || echo -e "\033[0;31m[ERROR] DLG File Not Found in the Output Directory.\033[0m"

### **Procedure 12: Archiving and Exporting Batch Molecular Docking Results**
---
This procedure compiles and compresses the primary publication-ready docking outputs across the entire batch dataset, specifically the structurally restored 3D poses (**.pdb**) and the simulation logs (**.dlg**), into a lightweight ZIP archive (**MolecularDockingOutput.zip**) for streamlined local downloading and downstream analysis.

Following the batch simulation execution (**Procedure 9**), thermodynamic evaluation (**Procedure 10**), and topological reconstruction (**Procedure 11**), the archive specifically targets:

1. **LigandName_BestPose.pdb:** The structurally restored 3D molecular poses for all evaluated ligands with complete covalent bond topology (**CONECT records**), ready for direct 2D/3D protein-ligand interaction profiling in software such as **BIOVIA Discovery Studio Visualizer**, **PyMOL**, or **UCSF Chimera**.

2. **LigandName_Result.dlg:** The primary simulation logs detailing full **Lamarckian Genetic Algorithm** parameters, cluster histograms, binding free energies ($\Delta G$), calculated inhibition constants ($K_i$), and **reference RMSD** metrics for each ligand.

**Export Instructions:**
1. **Option A (Compressed Archive Download):** Execute the cell below to generate **MolecularDockingOutput.zip** in the root execution environment (**`/content/`**). In the Google Colab left sidebar (**"Files"** icon), locate **MolecularDockingOutput.zip**, click the three vertical dots icon, and select **"Download"** option.

2. **Option B (Individual File Download):** Expand the **`MolecularDockingOutput`** directory in the left sidebar, right-click on any specific **LigandName_BestPose.pdb** and **LigandName_Result.dlg** file directly, and select **"Download"** option individually on each file until all the specific output files has been downloaded successfully.

To proceed, run/execute the **cell** below if you choose **Option A**.

Upon completion of this procedure, the entire molecular docking workflow is officially concluded, and the two primary output files (**LigandName_BestPose.pdb** and **LigandName_Result.dlg**) are fully prepared for further research analysis.

In [ ]:
# @title **Cell 1 Procedure 12**
!echo -e "\033[0;33m[INFORMATION] Compressing Output Directory into ZIP Archive...\033[0m" && zip -q -j /content/MolecularDockingOutput.zip /content/MolecularDockingOutput/*.pdb /content/MolecularDockingOutput/*.dlg && echo -e "\033[0;32m[SUCCESS] Output Directory Compression Complete.\033[0m" || echo -e "\033[0;31m[ERROR] Output Directory Compression Failed.\033[0m"

### **Procedure 13: Output Directory and ZIP Archive Purging Process**
---
This optional final procedure performs a runtime environment cleanup by purging temporary simulation outputs and compressed archives from the virtual machine. Executing the two code cells below carries out the following maintenance operations:

1. **Output Directory Cleaning (Cell 1):** Empties all generated output files (**.pdb**, **.dlg**, and **.xml**) stored inside the **`MolecularDockingOutput`** directory.
2. **ZIP Archive Removal (Cell 2):** Deletes the compressed archive files (**.zip**) from the root execution environment (**`/content/`**) to free up storage space and maintain a clean workstation.

Execute this procedure only **AFTER** you have successfully downloaded all required result files or the ZIP archive to your local computer in the previous **Procedure 12**.

To proceed, run/execute the **two cells** below sequentially.

In [ ]:
# @title **Cell 1 Procedure 13**
!echo -e "\033[0;33m[INFORMATION] Starting Output Directory Cleaning Process...\033[0m" && rm -rf /content/MolecularDockingOutput/* && echo -e "\033[0;32m[SUCCESS] Output Directory Cleaning Process Complete.\033[0m" || echo -e "\033[0;31m[ERROR] Output Directory Cleaning Process Failed.\033[0m"

In [ ]:
# @title **Cell 2 Procedure 13**
!echo -e "\033[0;33m[INFORMATION] Removing ZIP Archive from Root Environment...\033[0m" && rm -f /content/*.zip && echo -e "\033[0;32m[SUCCESS] ZIP Archive Removal from Root Environment Complete.\033[0m" || echo -e "\033[0;31m[ERROR] Failed to Removing ZIP Archive from Root Environment.\033[0m"

### **Information**
---
If you wish to initiate a new batch molecular docking simulation session using a different set of ligands or targets within the same active runtime session, you do **not** need to re-run the entire notebook from **Procedure 1**. The binary tools, system dependencies, and base directory structures (**Procedures 1–5**) are already fully configured in your environment.

To process a new dataset, simply follow these steps:
1. **Upload New Input Files:** Place all newly prepared ligand structures (**.pdbqt**), grid parameter files (**.gpf**), and the target macromolecule structure (**.pdbqt**) into the **`MolecularDockingInput`** directory.
2. **Execute Workflow:** Run **Procedure 6** through **Procedure 13** sequentially to automate folder setup, AutoGrid4 grid calculations, AutoDock-GPU sequential batch simulations, thermodynamic extraction, topology restoration, and results archiving.

### **Citations and References**
---
If you use this automated workflow or its outputs in your published research, please cite the following foundational software packages and their corresponding primary publications:

1. **AutoDock-GPU**  
   * **Primary Publication**: *Santos-Martins, D., Solis-Vasquez, L., Tillack, A. F., Sanner, M. F., Koch, A., & Forli, S. (2021). Accelerating AutoDock4 with GPUs and Gradient-Based Local Search. Journal of Chemical Theory and Computation, 17(2), 1060–1073. https://doi.org/10.1021/acs.jctc.0c01006.*  
   * **Software Repository**: Forli Lab, Scripps Research. AutoDock-GPU (**Version 1.6**), https://github.com/ccsb-scripps/AutoDock-GPU (Accessed September 2026).

2. **AutoDock Suite (AutoDock4, AutoGrid4, and AutoDockTools4 Program)**  
   * **Primary Publication**: *Morris, G. M., Huey, R., Lindstrom, W., Sanner, M. F., Belew, R. K., Goodsell, D. S., & Olson, A. J. (2009). AutoDock4 and AutoDockTools4: Automated docking with selective receptor flexibility. Journal of Computational Chemistry, 30(16), 2785–2791. https://doi.org/10.1002/jcc.21256.*  
   * **Software Package**: Forli Lab, Scripps Research. AutoDock Suite (**Version 4.2.6**), https://autodock.scripps.edu (Accessed September 2026).

3. **Open Babel**  
   * **Primary Publication**: *O’Boyle, N. M., Banck, M., James, C. A., Morley, C., Vandermeersch, T., & Hutchison, G. R. (2011). Open Babel: An open chemical toolbox. Journal of Cheminformatics, 3(1), 33. https://doi.org/10.1186/1758-2946-3-33.*
   * **Software Package**: Open Babel Development Team. Open Babel (**Version 3.1.1**), https://openbabel.org (Accessed September 2026).

### **AI Disclosure Statement and Author Accountability**
---
This computational workflow (**.ipynb**) was conceived, designed, and directed entirely by **Farhan Hidayat** as a **Researcher** from **Bandung**, **West Java**, **Indonesia**. The author retains full scientific accountability for the conceptual framework, methodological design, parameter selection, and overall integrity and reproducibility of this notebook. In compliance with international publishing guidelines on **Generative AI** tools, **Google Gemini AI** was employed as an interactive generative coding assistant to support the technical implementation of this project. Under the author's direct instruction and domain-specific logic, the AI translated the research workflow into functional Python and Linux/Bash scripts, assisted in syntax synthesis, and resolved execution errors. The AI acted strictly as an execution tool following the author's explicit procedural specifications. All scientific decisions, including computational workflow architecture, molecular preparation standards, parameter and force field selection, spatial grid configuration, search algorithms, execution and workflow logic, and quantitative data analysis protocols, were defined by the author. The author conducted all validation, output verification, and functional testing of the code. At no point did generative AI replace human critical thinking, experimental design, or scientific interpretation in this notebook.

### **Author and Contact Information**
---
My name is **Farhan Hidayat**, a Researcher in Computational Pharmacy, Molecular Biology, and In Silico Drug Discovery from Bandung, West Java, Indonesia. For further inquiries or potential academic discussions and collaborations, please feel free to reach out through Email (research.farhanhidayat@gmail.com), [LinkedIn](https://www.linkedin.com/in/hidayatfarhan/), [GitHub](https://github.com/HHIADNA), or [ORCID](https://orcid.org/0009-0007-5304-302X).